### [ex03] Numbers: big integers and floating point arithmetic

In [ ]:
import math

print(2 ** 200)  # integers have arbitrary precision
print(math.factorial(30))
print(7 / 2, 7 // 2, 7 % 2, -7 // 2)

In [ ]:
print(0.1 + 0.2 == 0.3)

print(f"{0.1:.30f}")
print(f"{0.1 + 0.2:.30f}")
print(f"{0.3:.30f}")

# see: https://0.30000000000000004.com

In [ ]:
xs = [0.1] * 10

acc = 0.0
for x in xs:
    acc += x

print(acc, acc == 1.0)
print(math.isclose(acc, 1.0))
print(math.fsum(xs), math.fsum(xs) == 1.0)
print(sum(xs), sum(xs) == 1.0)  # the result depends on the Python version (3.12 changed sum)

In [ ]:
def ex03_1(x1: float, x2: float, x3: float) -> bool:
    s1 = x1 + x2 + x3
    s2 = x3 + x2 + x1

    return s1 == s2


print(ex03_1(0.1, 0.2, 0.3))

In [ ]:
print(math.exp(700))

try:
    print(math.exp(710))
except OverflowError as err:
    print(f"OverflowError: {err}")

print(math.inf, -math.inf, math.inf - math.inf, math.nan == math.nan)

### [ex04] Names and objects: references, mutability, copies

In [ ]:
l1 = [1, 2, 3]
l2 = [1, 2, 3]
print(l1 == l2, l1 is l2)

In [ ]:
l3 = [1, 2, 3]
l4 = l3  # no copy: two names, one object
l4[0] = -1
print(f"{l3 = }, {l4 = }, {l3 is l4 = }")

In [ ]:
def f1(lst: list[int]) -> None:
    lst = [10, 20, 30]  # rebinds the local name


def f2(lst: list[int]) -> None:
    lst.append(4)  # mutates the object


l5 = [1, 2, 3]
f1(l5)
print(f"{l5 = }")

l6 = [1, 2, 3]
f2(l6)
print(f"{l6 = }")

In [ ]:
import copy

l7 = [1, [2, 3]]
l8 = l7.copy()  # shallow copy
l9 = copy.deepcopy(l7)

l7[1][0] = -2
print(f"{l7 = }, {l8 = }, {l9 = }")

In [ ]:
t1 = (1, 2, 3)
s1 = "abc"

try:
    t1[0] = 0
except TypeError as err:
    print(f"TypeError: {err}")

s2 = s1.upper()  # a new object; s1 is unchanged
print(f"{s1 = }, {s2 = }")

In [ ]:
def add_sample(x: float, batch: list[float] = []) -> list[float]:
    batch.append(x)
    return batch


print(add_sample(1.0))
print(add_sample(2.0))
print(add_sample.__defaults__)

In [ ]:
row = [0.0] * 3
grid1 = [row] * 2  # two references to the same row
grid2 = [[0.0] * 3 for _ in range(2)]

grid1[0][0] = 1.0
grid2[0][0] = 1.0
print(f"{grid1 = }")
print(f"{grid2 = }")

### [ex05] Collections: a first look at a text

In [ ]:
TEXT = """
Litwo! Ojczyzno moja! ty jesteś jak zdrowie;
Ile cię trzeba cenić, ten tylko się dowie,
Kto cię stracił. Dziś piękność twą w całej ozdobie
Widzę i opisuję, bo tęsknię po tobie.

Panno święta, co Jasnej bronisz Częstochowy
I w Ostrej świecisz Bramie! Ty, co gród zamkowy
Nowogródzki ochraniasz z jego wiernym ludem!
Jak mnie dziecko do zdrowia powróciłaś cudem
(Gdy od płaczącej matki, pod Twoją opiekę
Ofiarowany, martwą podniosłem powiekę;
I zaraz mogłem pieszo, do Twych świątyń progu
Iść za wrócone życie podziękować Bogu),
Tak nas powrócisz cudem na Ojczyzny łono.
Tymczasem przenoś moją duszę utęsknioną
Do tych pagórków leśnych, do tych łąk zielonych,
Szeroko nad błękitnym Niemnem rozciągnionych;
Do tych pól malowanych zbożem rozmaitem,
Wyzłacanych pszenicą, posrebrzanych żytem;
Gdzie bursztynowy świerzop, gryka jak śnieg biała,
Gdzie panieńskim rumieńcem dzięcielina pała,
A wszystko przepasane jakby wstęgą, miedzą
Zieloną, na niej z rzadka ciche grusze siedzą.
"""

In [ ]:
lines = TEXT.strip().splitlines()
print(len(lines))
print(lines[0])
print(lines[-1])
print(lines[1:3])

In [ ]:
PUNCTUATION = "!;,.()"

words = [w.strip(PUNCTUATION).lower() for w in TEXT.split()]
words = [w for w in words if w]  # drop empty strings

print(len(words), words[:8])

In [ ]:
unique_words = set(words)
print(len(unique_words))
print("zdrowie" in unique_words, "python" in unique_words)

In [ ]:
counts: dict[str, int] = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1

print(counts["do"], counts["tych"], counts["cudem"])

In [ ]:
top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:5]
print(top)

for word, n in top:
    print(f"{word:>6} {'#' * n}")

In [ ]:
lengths = {w: len(w) for w in unique_words}
longest = max(lengths, key=lengths.get)
print(longest, lengths[longest])

In [ ]:
from collections import Counter

print(Counter(words).most_common(5))

In [ ]:
bigrams = list(zip(words, words[1:]))
print(bigrams[:4])

after_do = Counter(nxt for prev, nxt in bigrams if prev == "do")
print(after_do.most_common())

### [ex06] Vectors and the nearest neighbour: a first model

In [ ]:
import math

Vector = tuple[float, float]

# (petal length [cm], petal width [cm]), species - a small sample of the Iris dataset
TRAIN: list[tuple[Vector, str]] = [
    ((1.4, 0.2), "setosa"),
    ((1.5, 0.2), "setosa"),
    ((1.7, 0.2), "setosa"),
    ((1.6, 0.2), "setosa"),
    ((1.3, 0.3), "setosa"),
    ((4.7, 1.4), "versicolor"),
    ((3.5, 1.0), "versicolor"),
    ((4.8, 1.8), "versicolor"),
    ((3.8, 1.1), "versicolor"),
    ((4.4, 1.2), "versicolor"),
    ((6.0, 2.5), "virginica"),
    ((5.1, 2.0), "virginica"),
    ((5.7, 2.3), "virginica"),
    ((6.1, 1.9), "virginica"),
    ((5.6, 2.4), "virginica"),
]

TEST: list[tuple[Vector, str]] = [
    ((1.7, 0.4), "setosa"),
    ((1.6, 0.2), "setosa"),
    ((1.4, 0.3), "setosa"),
    ((4.5, 1.3), "versicolor"),
    ((4.4, 1.4), "versicolor"),
    ((4.2, 1.2), "versicolor"),
    ((6.6, 2.1), "virginica"),
    ((6.0, 1.8), "virginica"),
    ((5.2, 2.3), "virginica"),
    ((4.8, 1.8), "virginica"),
]

In [ ]:
def dot(u: Vector, v: Vector) -> float:
    return sum(ui * vi for ui, vi in zip(u, v))


def norm(u: Vector) -> float:
    return math.sqrt(dot(u, u))


print(dot((1.0, 2.0), (3.0, 4.0)), norm((3.0, 4.0)))

In [ ]:
def distance(u: Vector, v: Vector) -> float:
    """The Euclidean distance between u and v."""
    raise NotImplementedError  # TODO


def predict(x: Vector, train: list[tuple[Vector, str]]) -> str:
    """The label of the training example nearest to x."""
    raise NotImplementedError  # TODO


def accuracy(test: list[tuple[Vector, str]], train: list[tuple[Vector, str]]) -> float:
    """The fraction of test examples whose predicted label is correct."""
    raise NotImplementedError  # TODO

In [ ]:
print(distance((0.0, 0.0), (3.0, 4.0)))  # expected: 5.0
print(predict((1.5, 0.3), TRAIN))  # expected: setosa
print(accuracy(TEST, TRAIN))  # expected: 0.9

### [ex07] Speed: loops, built-ins and NumPy

In [ ]:
from timeit import timeit
from typing import Final

N: Final[int] = 10_000_000

In [ ]:
def while_loop_based_sum(n: int = N) -> int:
    res = 0
    i = 0
    while i < n:
        res += i
        i += 1
    return res


def for_loop_based_sum(n: int = N) -> int:
    res = 0
    for i in range(n):
        res += i
    return res


def sum_range_based_sum(n: int = N) -> int:
    return sum(range(n))

In [ ]:
dt_while = timeit(while_loop_based_sum, number=3)
dt_for = timeit(for_loop_based_sum, number=3)
dt_sum = timeit(sum_range_based_sum, number=3)

print(f"{dt_while = :.2f} s, {dt_for = :.2f} s, {dt_sum = :.2f} s")

In [ ]:
try:
    import numpy as np
except ModuleNotFoundError:
    print("NumPy is not installed in this environment (it is available in Google Colab)")
else:
    def numpy_based_sum(n: int = N) -> int:
        return int(np.sum(np.arange(n, dtype=np.int64)))

    dt_np = timeit(numpy_based_sum, number=3)

    print(f"{dt_np = :.2f} s")
    print(f"{dt_while / dt_np = :.0f}, {dt_for / dt_np = :.0f}, {dt_sum / dt_np = :.0f}")
    print(while_loop_based_sum() == numpy_based_sum())